In [7]:
# -*- coding: utf-8 -*-
"""
泰坦尼克号生存预测
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    roc_auc_score, roc_curve, precision_recall_curve, auc as sklearn_auc,
    precision_score, recall_score, f1_score
)

# 1. 中文绘图全局配置
matplotlib.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'WenQuanYi Zen Hei'
]
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['savefig.bbox'] = 'tight'

# 字体测试
plt.figure()
plt.text(0.5, 0.5, '中文测试', fontsize=20, ha='center')
plt.close()
print("✅ 字体配置正常")

# 2. 读取本地数据集
try:
    train_df = pd.read_csv("train.csv", encoding="utf-8")
    print(f"\n✅ 成功加载本地数据集，共 {len(train_df)} 行数据")
except FileNotFoundError:
    print("❌ 错误：未找到 train.csv")
    print("💡 请将 train.csv 和代码放在同一个文件夹！")
    exit()

# 3. 数据预处理
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])
train_df = train_df.drop('Cabin', axis=1)

# 提取称呼
train_df['Title'] = train_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
title_mapping = {
    'Mr':'Mr','Mrs':'Mrs','Miss':'Miss','Master':'Master',
    'Dr':'Other','Rev':'Other','Col':'Other','Major':'Other',
    'Mlle':'Miss','Countess':'Other','Ms':'Miss','Lady':'Other',
    'Jonkheer':'Other','Don':'Other','Dona':'Other','Mme':'Mrs',
    'Capt':'Other','Sir':'Other'
}
train_df['Title'] = train_df['Title'].map(title_mapping)

# 构造新特征
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)

# 年龄、票价分组
train_df['AgeBin'] = pd.cut(train_df['Age'], bins=[0,12,18,35,60,100],
                            labels=['儿童','青少年','青年','中年','老年'])
train_df['FareBin'] = pd.qcut(train_df['Fare'], 4,
                              labels=['低票价','中低票价','中高票价','高票价'])

# 删除无用列
drop_cols = ['PassengerId','Name','Ticket','Age','Fare','SibSp','Parch']
train_df = train_df.drop(drop_cols, axis=1)

# 4. 划分数据集 & 搭建模型
X = train_df.drop('Survived', axis=1)
y = train_df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = ['Sex','Embarked','Title','AgeBin','FareBin']
numeric_features = ['Pclass','FamilySize','IsAlone']

# 预处理流水线
num_pipe = Pipeline(steps=[('scaler', StandardScaler())])
cat_pipe = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

model = Pipeline(steps=[
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        random_state=42
    ))
])

# 训练
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:,1]

# ====================== 实验报告关键指标输出 ======================
print("\n" + "="*60)
print("                泰坦尼克号生存预测 实验报告指标")
print("="*60)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print(f"准确率 (Accuracy): {acc:.4f}")
print(f"精确率 (Precision): {prec:.4f}")a
print(f"召回率 (Recall): {recall:.4f}")
print(f"F1分数 (F1-Score): {f1:.4f}")
print(f"AUC-ROC: {roc_auc:.4f}")

print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=['死亡', '存活']))

# 5折交叉验证
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"\n5折交叉验证准确率: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# ====================== 1. 增强版混淆矩阵 ======================
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)

annot_matrix = [
    [f"{cm[0,0]}\n(TN 真负例)", f"{cm[0,1]}\n(FP 假正例)"],
    [f"{cm[1,0]}\n(FN 假负例)", f"{cm[1,1]}\n(TP 真正例)"]
]

ax = sns.heatmap(
    cm, 
    annot=annot_matrix, 
    fmt='', 
    cmap='Blues',
    xticklabels=['预测：死亡', '预测：存活'],
    yticklabels=['真实：死亡', '真实：存活'],
    annot_kws={"size": 12},
    cbar=True
)

plt.title('泰坦尼克号生存预测模型混淆矩阵', fontsize=16, pad=20)
plt.xlabel('预测标签', fontsize=14, labelpad=15)
plt.ylabel('真实标签', fontsize=14, labelpad=15)
plt.tight_layout()
plt.savefig('confusion_matrix_report.png')
plt.close()
print("\n✅ 混淆矩阵已保存：confusion_matrix_report.png")

# ====================== 2. ROC曲线（直观模型区分能力） ======================
plt.figure(figsize=(8, 6))
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC曲线 (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='随机猜测')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('假阳性率 (FPR)', fontsize=12)
plt.ylabel('真阳性率 (TPR)', fontsize=12)
plt.title('泰坦尼克号生存预测模型 ROC 曲线', fontsize=16, pad=20)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png')
plt.close()
print("✅ ROC曲线已保存：roc_curve.png")

# ====================== 3. PR曲线（针对不平衡数据的评估） ======================
plt.figure(figsize=(8, 6))
precision, recall, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = sklearn_auc(recall, precision)
plt.plot(recall, precision, color='blue', lw=2, label=f'PR曲线 (AUC = {pr_auc:.4f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('召回率 (Recall)', fontsize=12)
plt.ylabel('精确率 (Precision)', fontsize=12)
plt.title('泰坦尼克号生存预测模型 PR 曲线', fontsize=16, pad=20)
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curve.png')
plt.close()
print("✅ PR曲线已保存：pr_curve.png")

# ====================== 4. 学习曲线（判断过拟合/欠拟合） ======================
plt.figure(figsize=(10, 6))
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=5, n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.plot(train_sizes, train_mean, color='blue', marker='o', markersize=5, 
         label='训练集准确率')
plt.fill_between(train_sizes, train_mean + train_std, train_mean - train_std, 
                 alpha=0.15, color='blue')
plt.plot(train_sizes, test_mean, color='green', linestyle='--', marker='s', 
         markersize=5, label='验证集准确率')
plt.fill_between(train_sizes, test_mean + test_std, test_mean - test_std, 
                 alpha=0.15, color='green')

plt.title('泰坦尼克号生存预测模型 学习曲线', fontsize=16, pad=20)
plt.xlabel('训练样本数', fontsize=12)
plt.ylabel('准确率', fontsize=12)
plt.grid(alpha=0.3)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig('learning_curve.png')
plt.close()
print("✅ 学习曲线已保存：learning_curve.png")

# ====================== 5. 特征重要性条形图 ======================
cat_enc = model.named_steps['prep'].named_transformers_['cat'].named_steps['onehot']
cat_names = cat_enc.get_feature_names_out(categorical_features)
all_names = np.concatenate([numeric_features, cat_names])
imp = model.named_steps['rf'].feature_importances_
idx = np.argsort(imp)[::-1]

plt.figure(figsize=(12, 7))
plt.bar(range(len(imp)), imp[idx], align='center')
plt.xticks(range(len(imp)), all_names[idx], rotation=90, fontsize=10)
plt.title('泰坦尼克号生存预测 特征重要性排序', fontsize=16, pad=20)
plt.ylabel('重要性得分', fontsize=12)
plt.xlabel('特征名称', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance_report.png')
plt.close()
print("✅ 特征重要性图已保存：feature_importance_report.png")

print("\n🎉 所有实验材料生成完成！")

✅ 字体配置正常

✅ 成功加载本地数据集，共 891 行数据

                泰坦尼克号生存预测 实验报告指标
准确率 (Accuracy): 0.7765
精确率 (Precision): 0.7377
召回率 (Recall): 0.6522
F1分数 (F1-Score): 0.6923
AUC-ROC: 0.8215

分类报告:
              precision    recall  f1-score   support

          死亡       0.80      0.85      0.82       110
          存活       0.74      0.65      0.69        69

    accuracy                           0.78       179
   macro avg       0.77      0.75      0.76       179
weighted avg       0.77      0.78      0.77       179


5折交叉验证准确率: 0.8093 (±0.0282)

✅ 混淆矩阵已保存：confusion_matrix_report.png
✅ ROC曲线已保存：roc_curve.png
✅ PR曲线已保存：pr_curve.png
✅ 学习曲线已保存：learning_curve.png
✅ 特征重要性图已保存：feature_importance_report.png

🎉 所有实验材料生成完成！
